In [1]:
##### INLAB

In [2]:
!pip install tensorflow

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

In [10]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("Libraries imported successfully!")

# Movie reviews
reviews = [
    "The product quality is excellent",
    "I am very happy with the service",
    "The delivery was very slow",
    "The product is damaged",
    "Excellent customer support",
    "Very disappointed with the purchase"
]

# Sentiment labels
labels = np.array([1, 1, 0, 0, 1, 0])

print("\nMovie Reviews:")
for i, review in enumerate(reviews, 1):
    sentiment = "Positive" if labels[i-1] == 1 else "Negative"
    print(f"{i}. {review} -> {sentiment}")

Libraries imported successfully!

Movie Reviews:
1. The product quality is excellent -> Positive
2. I am very happy with the service -> Positive
3. The delivery was very slow -> Negative
4. The product is damaged -> Negative
5. Excellent customer support -> Positive
6. Very disappointed with the purchase -> Negative


In [11]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(reviews)

sequences = tokenizer.texts_to_sequences(reviews)

print("Word Index:")
print(tokenizer.word_index)

print("\nInteger Encoded Sentences:")
for i, seq in enumerate(sequences, 1):
    print(f"{i}: {seq}")

Word Index:
{'the': 1, 'very': 2, 'product': 3, 'is': 4, 'excellent': 5, 'with': 6, 'quality': 7, 'i': 8, 'am': 9, 'happy': 10, 'service': 11, 'delivery': 12, 'was': 13, 'slow': 14, 'damaged': 15, 'customer': 16, 'support': 17, 'disappointed': 18, 'purchase': 19}

Integer Encoded Sentences:
1: [1, 3, 7, 4, 5]
2: [8, 9, 2, 10, 6, 1, 11]
3: [1, 12, 13, 2, 14]
4: [1, 3, 4, 15]
5: [5, 16, 17]
6: [2, 18, 6, 1, 19]


In [12]:
# Padding
max_length = max(len(seq) for seq in sequences)

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post"
)

print("Padded Sequences:")
print(X)

print("\nSequence Shape:", X.shape)
print("Maximum Sequence Length:", max_length)

Padded Sequences:
[[ 1  3  7  4  5  0  0]
 [ 8  9  2 10  6  1 11]
 [ 1 12 13  2 14  0  0]
 [ 1  3  4 15  0  0  0]
 [ 5 16 17  0  0  0  0]
 [ 2 18  6  1 19  0  0]]

Sequence Shape: (6, 7)
Maximum Sequence Length: 7


In [13]:
# Build LSTM model

vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 16

model = Sequential([
    Embedding(vocab_size, embedding_dim),
    LSTM(32),
    Dense(1, activation="sigmoid")
])

# Build the model using the input shape
model.build(input_shape=(None, max_length))

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 7, 16)               │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │           6,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,625 (25.88 KB)

 Trainable params: 6,625 (25.88 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Train the model
history = model.fit(
    X,
    labels,
    epochs=100,
    verbose=0
)

print("LSTM model trained successfully!")

LSTM model trained successfully!


In [15]:
# Predict sentiment
predictions = model.predict(X, verbose=0)

print("Sentiment Predictions:")
print("=" * 55)

for i, probability in enumerate(predictions, 1):

    sentiment = "Positive" if probability[0] >= 0.5 else "Negative"

    print(
        f"{i}. {reviews[i-1]}"
    )
    print(
        f"   Probability: {probability[0]:.4f}"
    )
    print(
        f"   Prediction : {sentiment}"
    )
    print()

Sentiment Predictions:
1. The product quality is excellent
   Probability: 0.9877
   Prediction : Positive

2. I am very happy with the service
   Probability: 0.9956
   Prediction : Positive

3. The delivery was very slow
   Probability: 0.0036
   Prediction : Negative

4. The product is damaged
   Probability: 0.0109
   Prediction : Negative

5. Excellent customer support
   Probability: 0.9967
   Prediction : Positive

6. Very disappointed with the purchase
   Probability: 0.0037
   Prediction : Negative



In [16]:
new_review = input("Enter a movie/product review: ")

new_sequence = tokenizer.texts_to_sequences([new_review])
new_padded = pad_sequences(
    new_sequence,
    maxlen=max_length,
    padding="post"
)

probability = model.predict(new_padded, verbose=0)[0][0]

if probability >= 0.5:
    sentiment = "Positive"
else:
    sentiment = "Negative"

print("\nReview:", new_review)
print("Probability:", round(float(probability), 4))
print("Sentiment:", sentiment)

Enter a movie/product review:  The second half of the movie is so disappointing



Review: The second half of the movie is so disappointing
Probability: 0.0115
Sentiment: Negative
